# Bansuri Coach — Quickstart Notebook

Use this to:
1. Install dependencies and set your API key
2. Test a single frame analysis (no camera needed — use a photo)
3. Launch the full live Gradio app

---
### Two modes available
| Mode | Command | Best for |
|------|---------|----------|
| **Polling** | `python live_flute_coach.py --mode webcam` | Any Gemini model, easier to debug |
| **Live API** | `python live_flute_coach.py --mode webcam --live` | Sub-second latency, needs live-capable model |

## Step 1 — Install dependencies

In [ ]:
!pip install google-genai gradio opencv-python mss sounddevice librosa python-dotenv --quiet
print('Done.')

## Step 2 — Configure API key and model

In [ ]:
import os
import sys
sys.path.insert(0, '.')

# Option A: set directly here
os.environ['GEMINI_API_KEY'] = 'YOUR_API_KEY_HERE'

# Option B: create a .env file with:
#   GEMINI_API_KEY=your_key
#   GEMINI_MODEL_POLLING=gemini-3.1-pro          ← set to actual model name
#   GEMINI_MODEL_LIVE=gemini-2.0-flash-live-001   ← for live streaming

# Verify
from config import GEMINI_API_KEY, GEMINI_MODEL_POLLING, GEMINI_MODEL_LIVE
print(f'API key set  : {"YES" if GEMINI_API_KEY != "YOUR_API_KEY_HERE" else "NO — set it above"}')
print(f'Polling model: {GEMINI_MODEL_POLLING}')
print(f'Live model   : {GEMINI_MODEL_LIVE}')

## Step 3 — Test: analyse a single image frame

In [ ]:
import cv2
import numpy as np
import json
from live_flute_coach import PollingCoach, encode_frame_to_jpeg

# Option 1: take a photo with your webcam right now
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

# Option 2: load a still image
# frame = cv2.imread('test_photo.jpg')

if frame is None:
    print('No camera found. Load a test image instead.')
else:
    print(f'Captured frame: {frame.shape[1]}x{frame.shape[0]}')

    # Show it
    from PIL import Image
    import IPython.display as ipd
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    ipd.display(Image.fromarray(rgb).resize((640, 360)))

In [ ]:
# Send to Gemini — Lesson 1
coach = PollingCoach(lesson_num='1')
result = coach.analyse_frame(frame)

print('=== Gemini Feedback (Lesson 1) ===')
print(json.dumps(result, indent=2))

In [ ]:
# Send to Gemini — Lesson 2
coach2 = PollingCoach(lesson_num='2')
result2 = coach2.analyse_frame(frame)

print('=== Gemini Feedback (Lesson 2) ===')
print(json.dumps(result2, indent=2))

## Step 4 — Check which Gemini models you have access to

In [ ]:
from google import genai
from config import GEMINI_API_KEY

client = genai.Client(api_key=GEMINI_API_KEY)

print('Available models with generateContent support:')
for m in client.models.list():
    supported = getattr(m, 'supported_actions', []) or []
    if 'generateContent' in str(supported) or not supported:
        print(f'  {m.name}')

## Step 5 — Launch the Live Gradio App

Pick your mode below. The app opens in your browser automatically.

In [ ]:
# ── Polling mode — webcam — works with any model ──────────────────────────────
from live_flute_coach import build_gradio_app

app = build_gradio_app(capture_mode='webcam', use_live_api=False)
app.launch(inbrowser=True)

In [ ]:
# ── Polling mode — SCREEN SHARE — like AI Studio ──────────────────────────────
# This captures your primary monitor. Play a video of yourself on screen
# and Gemini will analyse it in real-time.

from live_flute_coach import build_gradio_app

app = build_gradio_app(capture_mode='screen', use_live_api=False)
app.launch(inbrowser=True)

In [ ]:
# ── Live API mode — lowest latency — needs live-capable model ─────────────────
# Set GEMINI_MODEL_LIVE in config.py or .env to a live-capable model first.
# e.g. gemini-2.0-flash-live-001

from live_flute_coach import build_gradio_app

app = build_gradio_app(capture_mode='webcam', use_live_api=True)
app.launch(inbrowser=True)

## Step 6 — Or run from terminal (recommended for production)

```bash
# Webcam, polling, Lesson 1
python live_flute_coach.py --lesson 1 --mode webcam

# Screen share, polling, Lesson 2  
python live_flute_coach.py --lesson 2 --mode screen

# Webcam, Live API (sub-second latency)
python live_flute_coach.py --lesson 1 --mode webcam --live
```

---
## Architecture Summary

```
Webcam / Screen
      │
      ▼
Frame capture thread (OpenCV / mss)
      │
      ├── JPEG encode
      │
      ├── [Polling] → Gemini generateContent API → JSON feedback
      │              (any model, every 1.5s)
      │
      └── [Live]   → Gemini Live API WebSocket → real-time text
                     (live-capable model, sub-second)
      │
Mic ──┤
      │
      ▼
PYIN pitch detection (parallel) ────► note name + Hz
      │
      ▼
Gradio UI — video feed + feedback + lesson status
```

### Setting the right model name
The model name for Gemini changes with each release. To find what you have access to, run Step 4 above.
Update `GEMINI_MODEL_POLLING` and `GEMINI_MODEL_LIVE` in `config.py` or `.env`.